# 2. Compute the score

Read the saved CSV tables, inspect the selected periods and peers, then calculate growth, Social, IR, Trend and raw FMI. Intermediate tables are saved under `../data/score/`. No database calls are made.

In [ ]:
import os
from pathlib import Path
from runpy import run_path
from statistics import median

import pandas as pd
from dotenv import load_dotenv

from aily_project.services.fmi import dossier_selection, social
from aily_project.services.fmi.configuration import input_policy
from aily_project.services.fmi.financial import comparable
from aily_project.services.fmi.inputs import read_inputs
from aily_project.services.fmi.models import (
    FMIResult,
    instant,
    truth,
)
from aily_project.services.fmi.policy import Policy
from aily_project.services.fmi.scenario import select_period, subject_keys

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
FMI = ROOT / "scripts/fashion-momentum-index"
load_dotenv(ROOT / ".env", override=False)
INPUTS_PATH = Path(
    os.environ.get("FMI_INPUTS_PATH", str(FMI / "inputs/FMI_inputs.json"))
)
DATA = Path(os.environ.get("FMI_DATA_DIR", str(FMI / "data")))
OUT = Path(os.environ.get("FMI_OUTPUT_DIR", str(ROOT / "outputs/fmi")))

# Small file-reading, CSV-saving and read-only query helpers.
io = run_path(str(FMI / "helpers.py"))
save_table, load_table = io["save_table"], io["load_table"]
read_source, read_snapshot = io["read_source"], io["read_snapshot"]

inputs = read_inputs(INPUTS_PATH)
policy = input_policy(INPUTS_PATH, Policy())
print("Input:", INPUTS_PATH)
print("Data:", DATA)
print("Delivery:", OUT)


EXTRACT, SCORE = DATA / "extract", DATA / "score"
snapshot = read_snapshot(EXTRACT)
cutoff = instant(snapshot.metadata["extracted_at"])
run_id = snapshot.metadata["extracted_at"]
entities = {e["entity_id"]: e for e in inputs.entities}
print("Saved extraction:", cutoff)
print("This run:", run_id)

## 2.1 One selected analysis per prospect

Financial subjects are routed only by an explicit dossier `source_entity_id`.
Missing or ambiguous analyses remain visible.

In [ ]:
results, selections, subjects = [], {}, {}
for entity in inputs.entities:
    if not truth(entity.get("prospect")):
        continue
    chosen = [
        a
        for a in inputs.analyses
        if a["entity_id"] == entity["entity_id"]
        and truth(a.get("selected"))
        and truth(a.get("headline"))
    ]
    analysis = chosen[0] if len(chosen) == 1 else None
    result = FMIResult(
        entity["entity_id"], analysis["analysis_id"] if analysis else None
    )
    result.values.update(
        **{
            k: entity.get(k)
            for k in (
                "name",
                "event_category",
                "event_invited",
                "fmi_required",
                "handle",
                "profile_id",
            )
        },
        run_id=run_id,
        policy_id=policy.version,
        headline_selection=bool(analysis),
        calculation_mode="dossier",
        release_status="dossier_defined_current_source_calculation",
        historical_status="retrospective_not_point_in_time_replay",
        data_extracted_at=cutoff,
        scenario_as_of=cutoff,
        ir_weight=policy.ir_weight,
        social_weight=policy.social_weight,
        trend_weight=policy.trend_weight,
        scenario_quarter_tolerance_days=policy.scenario_quarter_tolerance_days,
        scenario_forecast_tolerance_days=policy.scenario_forecast_tolerance_days,
    )
    if not analysis:
        result.gaps.append("analysis:one_selected_headline_required")
    else:
        selections[result.brand_id] = analysis
        subject = dossier_selection.source_subject(entity, analysis, inputs.entities)
        subjects[result.brand_id] = subject
        definition = (
            analysis["dossier"]
            .get("financial", {})
            .get("entities", {})
            .get(result.brand_id, {})
        )
        result.values.update(
            **{
                k: subject.get(k)
                for k in (
                    "light_code",
                    "ticker",
                    "shared_symbol",
                    "proxy",
                    "proxy_label",
                )
            }
        )
        result.values.update(
            financial_subject=definition.get("subject"),
            scope=definition.get("scope"),
            dossier_source=analysis["dossier"].get("source"),
            financial_basis="dossier",
            analysis_review_status="business_dossier_parameters",
            dossier_parameter_status=analysis["dossier"].get("status"),
            financial_selection_basis="dossier_dates_scope_metric_no_latest_fallback",
            comparability_status="dossier_defined_periods_and_scopes",
            financial_scope_status="source_scope_must_match_dossier",
            financial_unit_status="verified_source_units_required",
            revenue_alignment_status="explicit_dossier_periods_independent_of_forecast",
            parent_fallback_status="explicit_dossier_source_subject"
            if definition.get("source_entity_id")
            else "dossier_brand_subject",
            **{
                k: analysis.get(k)
                for k in (
                    "route",
                    "collection",
                    "show_date",
                    "timezone",
                    "forecast_target_label",
                    "forecast_target_start",
                    "forecast_target_end",
                )
            },
        )
        result.values.update(
            headline_method=analysis.get("headline_method", "dossier_reference"),
            dossier_score_basis=analysis["dossier"].get("score_basis"),
            dossier_headline_replication_status="original_reference_cohort_unavailable"
            if analysis["dossier"].get("score_basis") == "percentile"
            else "current_measurements_not_frozen_reference_values",
        )
        if (
            analysis["dossier"].get("score_basis") == "percentile"
            and analysis.get("headline_method", "dossier_reference")
            != "weighted_components"
        ):
            result.gaps.append(
                "dossier:percentile_normalization_requires_original_reference_cohort"
            )
    results.append(result)
display(
    save_table([r.record() for r in results], SCORE / "analyses")[
        ["brand_id", "name", "collection", "financial_subject", "scope"]
    ]
)

## 2.2 Select the three brand observations

The selector verifies exact dossier periods, scope, currency, units and source provenance.
It supports evidenced annual aggregation only from four verified gapless quarters.
Partial selections are retained for inspection, but growth requires a complete comparable series.

In [ ]:
brand_series, period_checks, selected_periods = {}, [], []
for result in results:
    if result.brand_id not in selections:
        continue
    analysis = selections[result.brand_id]
    periods, reasons = dossier_selection.series(
        inputs, entities[result.brand_id], snapshot, analysis
    )
    result.financial_periods.extend(periods)
    result.gaps.extend(reasons)
    brand_series[result.brand_id] = {p["role"]: p for p in periods}
    result.values["financial_coverage_status"] = (
        "verified_dossier_measurements"
        if periods and not reasons
        else "missing_verified_dossier_measurements"
    )
    if periods and not reasons:
        result.values.update(
            financial_scope_status="source_scope_matches_dossier",
            financial_unit_status="verified_selected_source_units",
        )
    selected_periods.extend(periods)
    period_checks.append(
        {
            "brand_id": result.brand_id,
            "selected_periods": len(periods),
            "complete_comparable_series": not reasons and comparable(periods),
            "missing_reasons": "; ".join(reasons),
        }
    )
    for period in periods:
        role = period["role"]
        for source, target in (
            ("amount", "revenue"),
            ("period_label", "period_label"),
            ("period_start", "period_start"),
            ("period_end", "period_end"),
            ("currency", "currency"),
            ("unit_multiplier", "unit_multiplier"),
            ("source_classification", "source_classification"),
            ("source_row_id", "source_row_id"),
            ("source_relation", "source_relation"),
            ("source_url", "source_url"),
        ):
            result.values[f"{role}_{target}"] = period.get(source)
        result.values.update(
            currency=period["currency"], unit_multiplier=period["unit_multiplier"]
        )
        result.values["financial_currency_status"] = "source_currency_or_unknown"
        # Context only: margins never affect FMI. Require the exact revenue basis.
        for metric in ("cost_of_revenue", "gross_profit", "ebitda"):
            context, _ = select_period(
                subjects[result.brand_id],
                snapshot,
                period["period_end"],
                role,
                analysis["analysis_id"],
                0,
                metric,
            )
            native = next(
                (
                    r
                    for r in snapshot.financials
                    if context
                    and r.get("source_row_id") == context["source_row_id"]
                    and r.get("metric") == metric
                ),
                {},
            )
            same_semantics = all(
                native.get(k) == period.get(k)
                for k in ("scope", "currency", "unit_multiplier")
            )
            same_period = context and all(
                context.get(k) == period.get(k)
                for k in (
                    "source_relation",
                    "scope",
                    "currency",
                    "unit_multiplier",
                    "period_start",
                    "period_end",
                )
            )
            if same_semantics and same_period:
                result.values[f"{role}_{metric}"] = context["amount"]
                result.values[f"{role}_{metric}_margin"] = (
                    context["amount"] / period["amount"]
                )
                result.financial_periods.append(context)
    if not reasons and comparable(periods):
        p = brand_series[result.brand_id]
        result.values["earlier_growth"] = (
            p["earlier"]["amount"] / p["oldest"]["amount"] - 1
        )
        result.values["latest_growth"] = (
            p["latest"]["amount"] / p["earlier"]["amount"] - 1
        )
display(save_table(period_checks, SCORE / "financial-selection-checks"))
display(save_table(selected_periods, SCORE / "selected-financial-periods").head(30))

## 2.3 Use the same usable peer subset twice

One assigned usable peer is sufficient; several use the median.
Exclude missing/conflicting three-period series and duplicate/self financial subjects.
Do not replace an excluded peer with an unassigned company.

In [ ]:
peer_checks = []
for result in results:
    if result.brand_id not in selections:
        continue
    analysis = selections[result.brand_id]
    seen, used = subject_keys(subjects[result.brand_id]), []
    assigned = [p for p in inputs.peers if p["analysis_id"] == analysis["analysis_id"]]
    for peer in assigned:
        entity = entities[peer["peer_entity_id"]]
        subject = dossier_selection.source_subject(entity, analysis, inputs.entities)
        periods, reasons = dossier_selection.series(inputs, entity, snapshot, analysis)
        keys = subject_keys(subject)
        if keys & seen:
            reasons.append("duplicate_or_self_financial_subject")
        if not comparable(periods):
            reasons.append("missing_comparable_three_quarter_series")
        p = {row["role"]: row for row in periods}
        earlier = (
            p["earlier"]["amount"] / p["oldest"]["amount"] - 1
            if comparable(periods)
            else None
        )
        latest = (
            p["latest"]["amount"] / p["earlier"]["amount"] - 1
            if comparable(periods)
            else None
        )
        check = {
            **peer,
            "brand_id": result.brand_id,
            "peer_name": subject["name"],
            "financial_subject": subject.get("financial_subject"),
            "proxy_label": subject.get("proxy_label"),
            "earlier_growth": earlier,
            "latest_growth": latest,
            "status": "excluded" if reasons else "dossier_periods_used",
            "missing_reasons": "; ".join(reasons),
            **{
                f"{row['role']}_{key}": row.get(key)
                for row in periods
                for key in ("period_label", "period_end")
            },
        }
        result.peers.append(check)
        peer_checks.append(check)
        if not reasons:
            seen.update(keys)
            used.append(check)
            result.financial_periods.extend(periods)
    v = result.values
    approved = [p for p in assigned if truth(p.get("approved"))]
    v.update(
        approved_peer_count=len(approved),
        approved_peer_names="; ".join(
            entities[p["peer_entity_id"]]["name"] for p in approved
        ),
        peer_review_status="dossier_defined_usable_peer_subset",
        used_peer_count=len(used),
        used_peer_names="; ".join(p["peer_name"] for p in used),
        used_peer_financial_subjects="; ".join(
            f"{p['peer_name']}: {p.get('financial_subject') or p['peer_name']}"
            for p in used
        ),
        peer_exclusion_reasons="; ".join(
            f"{p['peer_name']}: {p['missing_reasons']}"
            for p in result.peers
            if p["status"] == "excluded"
        ),
        used_peer_periods="; ".join(
            f"{p['peer_name']}: {p.get('oldest_period_label')} "
            f"ending {p.get('oldest_period_end')}, "
            f"{p.get('earlier_period_label')} ending {p.get('earlier_period_end')}, "
            f"{p.get('latest_period_label')} ending {p.get('latest_period_end')}"
            for p in used
        ),
        peer_calendar_alignment_status="different_native_end_dates"
        if any(p.get("latest_period_end") != v.get("latest_period_end") for p in used)
        else "same_native_end_dates"
        if used
        else "no_usable_peers",
    )
    if not used:
        result.gaps.append(
            "peers:no_assigned_peer_with_comparable_three_quarter_series"
        )
    for role in ("earlier", "latest"):
        if used:
            benchmark = median(p[f"{role}_growth"] for p in used)
            v[f"{role}_peer_median_growth"] = benchmark
            if v.get(f"{role}_growth") is not None:
                v[f"{role}_gap_pp"] = 100 * (v[f"{role}_growth"] - benchmark)
display(save_table(peer_checks, SCORE / "peer-selection"))
growth_columns = [
    "brand_id",
    "name",
    "earlier_growth",
    "latest_growth",
    "used_peer_count",
    "used_peer_names",
    "earlier_peer_median_growth",
    "latest_peer_median_growth",
    "earlier_gap_pp",
    "latest_gap_pp",
]
display(
    save_table(
        [{k: r.record().get(k) for k in growth_columns} for r in results],
        SCORE / "growth-comparisons",
    )
)

## 2.4 Social windows, usable posts and medians

The existing selection helper handles timezone-qualified captures, cutoff-before-deduplication,
caption rules, human exclusions, sample minimums and common-month filtering.
Inspect included **and** excluded posts below. Missing likes/comments stay missing.

For a single account, lift is current median / baseline median − 1.
Portfolio dossiers use the median of eligible member lifts, never pooled likes.
TikTok and undefined relevance rules remain unavailable.

In [ ]:
social_checks, social_posts = [], []
for result in results:
    if result.brand_id not in selections:
        continue
    analysis = selections[result.brand_id]
    settings = analysis["dossier"].get("social", {})
    values, posts, reasons = social.calculate(
        inputs,
        snapshot,
        analysis,
        entities[result.brand_id],
        cutoff,
        policy,
        simulation=True,
    )
    # Recalculate the visible headline from the validated samples/member lifts.
    metric = settings.get("metric", "likes")
    lift = values.get(f"{metric}_lift")
    if not reasons and lift is not None:
        values["social_score"] = max(0, min(100, 50 + 100 * lift))
    result.values.update(values)
    result.values["social_selection_basis"] = (
        "explicit_dossier_windows_and_relevance_rule"
    )
    result.values["social_capture_count"] = sum(
        str(p.get("profile_username", "")).lower().lstrip("@")
        == str(entities[result.brand_id].get("handle") or "").lower().lstrip("@")
        for p in snapshot.posts
    )
    result.social_posts = posts
    result.gaps.extend(reasons)
    social_posts.extend({"brand_id": result.brand_id, **p} for p in posts)
    social_checks.append(
        {
            "brand_id": result.brand_id,
            "metric": metric,
            "aggregation": settings.get("aggregation", "single_account"),
            "caption_pattern": settings.get("caption_pattern"),
            "capture_cutoff": settings.get("capture_cutoff") or cutoff,
            **values,
            "missing_reasons": "; ".join(reasons),
        }
    )
display(
    save_table(social_checks, SCORE / "social-samples").filter(
        regex="brand_id|metric|window|median|usable_posts|lift|score|missing_reasons"
    )
)
post_table = save_table(social_posts, SCORE / "social-post-selection")
if not post_table.empty:
    display(
        post_table.groupby(["brand_id", "sample", "inclusion_reason"])
        .size()
        .rename("posts")
    )
    display(post_table.head(25))

## 2.5 IR and dossier-specific Trend

IR = clip(50 + multiplier × latest peer-gap percentage points, 0, 100).

Trend preserves the dossier's rule and basis. It may use peer-gap change,
company-growth change, or an explicitly labelled assumption. A ±2pp boundary
is stable unless the configured policy says otherwise.

In [ ]:
for result in results:
    if result.brand_id not in selections:
        continue
    v = result.values
    settings = selections[result.brand_id]["dossier"].get("trend", {})
    v["trend_basis"], v["trend_comparison"] = (
        settings.get("basis"),
        settings.get("comparison"),
    )
    if v.get("latest_gap_pp") is not None:
        v["ir_score"] = max(
            0, min(100, 50 + policy.ir_gap_multiplier * v["latest_gap_pp"])
        )
    if settings.get("basis") == "assumed":
        v["trend_score"], v["trend_label"] = (
            settings["assumed_score"],
            "dossier_assumed_not_measured",
        )
    elif v.get("earlier_gap_pp") is not None and v.get("latest_gap_pp") is not None:
        earlier, latest = v["earlier_gap_pp"], v["latest_gap_pp"]
        if settings.get("basis") == "company_growth":
            earlier, latest = 100 * v["earlier_growth"], 100 * v["latest_growth"]
        change = latest - earlier
        v["trend_change_pp"] = change
        if change > policy.trend_change_boundary_pp:
            v["trend_score"], v["trend_label"] = (
                (80, "strong")
                if settings.get("rule") == "improvement_sign" and latest >= 0
                else (70, "up_and_coming")
            )
        elif change < -policy.trend_change_boundary_pp:
            v["trend_score"], v["trend_label"] = 30, "declining"
        elif (
            settings.get("rule") != "improvement_sign"
            and min(earlier, latest) >= policy.sustained_gap_pp
        ):
            v["trend_score"], v["trend_label"] = 80, "strong"
        else:
            v["trend_score"], v["trend_label"] = 50, "stable"
    if v.get("ir_score") is None:
        result.gaps.append("ir:missing_brand_or_peer_quarter_growth")
    if v.get("trend_score") is None:
        result.gaps.append("trend:missing_earlier_and_latest_quarter_gaps")
display(
    pd.DataFrame([r.record() for r in results]).filter(
        regex="brand_id|ir_score|trend_score|trend_basis|trend_change|trend_label"
    )
)

## 2.6 Weighted raw score and missing reasons

Each component keeps its own status and weighted contribution.
A complete raw FMI requires all three components **and** no unresolved FMI prerequisites.
The headline is calculated once per brand, not separately for each peer.

In [ ]:
for result in results:
    v = result.values
    for component in ("ir", "social", "trend"):
        score = v.get(f"{component}_score")
        v[f"{component}_status"] = "available" if score is not None else "unavailable"
        v[f"{component}_contribution"] = (
            score * v[f"{component}_weight"] if score is not None else None
        )
    complete = not result.gaps and all(
        v.get(f"{c}_score") is not None for c in ("ir", "social", "trend")
    )
    v["fmi_status"] = "available" if complete else "unavailable"
    if complete:
        raw = sum(v[f"{c}_contribution"] for c in ("ir", "social", "trend"))
        v["raw_fmi"] = raw
        v["raw_classification"] = (
            "High"
            if raw >= 70
            else "Medium"
            if raw >= 60
            else "Neutral"
            if raw >= 40
            else "Low"
        )
    v["evidence_refs"] = str(
        selections.get(result.brand_id, {}).get("dossier", {}).get("source") or ""
    )
score_rows = [r.record() for r in results]
display(
    save_table(score_rows, SCORE / "components").filter(
        regex="brand_id|name$|score$|contribution$|raw_fmi|fmi_status|missing_reasons"
    )
)
print(
    len(results),
    "prospects;",
    sum(r.values["fmi_status"] == "available" for r in results),
    "complete raw scores",
)
print("Next: 03_delivery.ipynb reads", SCORE / "components.csv")